# E3: human-labelled disputed frames -> first trusted pose accuracy

Part A builds a small CVAT package from the frames where ViTPose and YOLO11s-pose disagreed most in E0. All of them
come from held-out blocks, never used for training. Part B (after a person labels them in CVAT) measures PCK for
ViTPose-L SwimXYZ, pretrained YOLO11s-pose and the distilled YOLO11s-pose against **human** ground truth.

The set is deliberately the hardest cases (head-down bodies, splash), so absolute scores will be low. The comparison
between models on these cases is the point.


## 0. Drive

In [2]:
# 0. Drive, paths
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')
OUTD = DRIVE / 'pose_baseline'
ONNX = OUTD / 'models' / 'vitpose-l-swimxyz-mv.onnx'
YOLO_PT = OUTD / 'models' / 'yolo11s-pose.pt'
DIST_PT = OUTD / 'runs_pose' / 'yolo11s_pose_distill_960' / 'weights' / 'best.pt'
DISPUTED = OUTD / 'e0' / 'e0_disputed_heldout.csv'
DS_ZIP = DRIVE / 'datasets (1).zip'
E3 = OUTD / 'e3'
E3.mkdir(exist_ok=True)
for f in (ONNX, YOLO_PT, DIST_PT, DISPUTED, DS_ZIP):
    print('OK ' if f.is_file() else 'MISSING', f)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK  /content/drive/MyDrive/pose_baseline/models/vitpose-l-swimxyz-mv.onnx
OK  /content/drive/MyDrive/pose_baseline/models/yolo11s-pose.pt
OK  /content/drive/MyDrive/pose_baseline/runs_pose/yolo11s_pose_distill_960/weights/best.pt
OK  /content/drive/MyDrive/pose_baseline/e0/e0_disputed_heldout.csv
OK  /content/drive/MyDrive/datasets (1).zip


## 1. Install

In [3]:
# 1. Install
!pip -q install onnxruntime-gpu ultralytics
!mkdir -p /content/pose_tools


## 2. Tools

In [4]:
%%writefile /content/pose_tools/pose_common.py
"""Shared helpers for the pose proposal / zero-shot tools (no project imports).

Top-down model contract (from the ViTPose-L SwimXYZ model card): input (N, 3, 256, 192) float32,
ImageNet-normalised RGB crop; output (N, 17, 64, 48) heatmaps in COCO-17 order.
Pure-numpy parts (box->crop parameters, heatmap decoding, PCK) are testable without weights.
"""
from __future__ import annotations

import math
from typing import Any

import numpy as np

COCO17 = ["nose", "left_eye", "right_eye", "left_ear", "right_ear", "left_shoulder", "right_shoulder",
          "left_elbow", "right_elbow", "left_wrist", "right_wrist", "left_hip", "right_hip",
          "left_knee", "right_knee", "left_ankle", "right_ankle"]
UPPER_BODY = [0, 5, 6, 7, 8, 9, 10]  # nose, shoulders, elbows, wrists
COCO_SKELETON = [[16, 14], [14, 12], [17, 15], [15, 13], [12, 13], [6, 12], [7, 13], [6, 7], [6, 8],
                 [7, 9], [8, 10], [9, 11], [2, 3], [1, 2], [1, 3], [2, 4], [3, 5], [4, 6], [5, 7]]
IN_W, IN_H = 192, 256
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def crop_params(box_xyxy: tuple[float, float, float, float], scale: float = 1.25) -> tuple[float, float, float]:
    """Return (x0, y0, s) of the crop window: window covers IN_W/s x IN_H/s pixels, box centred, padded."""
    x1, y1, x2, y2 = box_xyxy
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    bw, bh = max(x2 - x1, 1.0) * scale, max(y2 - y1, 1.0) * scale
    if bw / bh > IN_W / IN_H:
        bh = bw * IN_H / IN_W
    else:
        bw = bh * IN_W / IN_H
    s = IN_W / bw
    return cx - bw / 2, cy - bh / 2, s


def preprocess(frame_bgr: np.ndarray, box_xyxy: tuple[float, float, float, float]
               ) -> tuple[np.ndarray, tuple[float, float, float]]:
    import cv2  # lazy: Colab has it

    x0, y0, s = crop_params(box_xyxy)
    m = np.array([[s, 0, -x0 * s], [0, s, -y0 * s]], dtype=np.float32)
    crop = cv2.warpAffine(frame_bgr, m, (IN_W, IN_H), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    rgb = crop[:, :, ::-1].astype(np.float32) / 255.0
    rgb = (rgb - MEAN) / STD
    return rgb.transpose(2, 0, 1)[None], (x0, y0, s)


def decode_heatmaps(hm: np.ndarray, params: tuple[float, float, float]) -> np.ndarray:
    """hm: (17, H, W) -> (17, 3) [x, y, score] in image pixels (argmax, +quarter-pixel shift)."""
    k, h, w = hm.shape
    x0, y0, s = params
    out = np.zeros((k, 3), dtype=np.float32)
    for i in range(k):
        flat = int(np.argmax(hm[i]))
        iy, ix = divmod(flat, w)
        score = float(hm[i, iy, ix])
        fx, fy = float(ix), float(iy)
        if 0 < ix < w - 1:
            fx += 0.25 * np.sign(hm[i, iy, ix + 1] - hm[i, iy, ix - 1])
        if 0 < iy < h - 1:
            fy += 0.25 * np.sign(hm[i, iy + 1, ix] - hm[i, iy - 1, ix])
        cx, cy = (fx + 0.5) * IN_W / w, (fy + 0.5) * IN_H / h  # crop pixels
        out[i] = (cx / s + x0, cy / s + y0, score)
    return out


class OnnxVitPose:
    """Top-down ONNX pose model (ViTPose-L SwimXYZ export). Input name is read from the graph."""

    def __init__(self, path: str, providers: list[str] | None = None) -> None:
        import onnxruntime as ort

        self.sess = ort.InferenceSession(path, providers=providers or ["CUDAExecutionProvider",
                                                                         "CPUExecutionProvider"])
        self.input_name = self.sess.get_inputs()[0].name

    def predict(self, frame_bgr: np.ndarray, boxes: list[tuple[float, float, float, float]]) -> list[np.ndarray]:
        res = []
        for b in boxes:
            x, params = preprocess(frame_bgr, b)
            hm = self.sess.run(None, {self.input_name: x})[0][0]
            res.append(decode_heatmaps(hm, params))
        return res


def iou(a: tuple[float, ...], b: tuple[float, ...]) -> float:
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def pck(pred: np.ndarray, gt: np.ndarray, box_xyxy: tuple[float, ...], thr: float,
        idx: list[int] | None = None) -> tuple[int, int]:
    """(correct, total) over GT keypoints with visibility > 0. Normaliser: longer side of the GT box."""
    norm = max(box_xyxy[2] - box_xyxy[0], box_xyxy[3] - box_xyxy[1], 1.0)
    ok = tot = 0
    for i in idx if idx is not None else range(len(gt)):
        if gt[i][2] <= 0:
            continue
        tot += 1
        if math.hypot(pred[i][0] - gt[i][0], pred[i][1] - gt[i][1]) <= thr * norm:
            ok += 1
    return ok, tot


def confident_upper(pred: np.ndarray, min_conf: float = 0.5) -> int:
    return int(sum(1 for i in UPPER_BODY if pred[i][2] >= min_conf))


def as_info(**kw: Any) -> dict[str, Any]:
    return kw


Overwriting /content/pose_tools/pose_common.py


In [9]:
%%writefile /content/pose_tools/zero_shot_eval.py
"""Zero-shot comparison of pose models on HAND-CORRECTED frames (CVAT export, COCO keypoints 1.0).

Metrics (all on GT keypoints with visibility > 0; normaliser = longer side of the GT box):
  PCK@0.05 / 0.1 / 0.2 on upper body (nose, shoulders, elbows, wrists) and on all 17,
  share of frames with >= 3 confident (>=0.5) upper-body keypoints, latency per person.
Models (any subset):
  --vitpose-onnx   ViTPose-L SwimXYZ ONNX, top-down on the GT box
  --yolo-pose      Ultralytics pose weights, full frame; the prediction with best IoU to the GT box is used
Only run on the `real_val` frames; select with --split-manifest + --split real_val. Report per camera type via --group.
"""
from __future__ import annotations

import argparse
import csv
import json
import time
from pathlib import Path

import cv2
import numpy as np

from pose_common import UPPER_BODY, OnnxVitPose, confident_upper, iou, pck

THRS = (0.05, 0.1, 0.2)


def load_gt(coco_path: Path) -> dict[str, tuple[np.ndarray, tuple[float, float, float, float]]]:
    d = json.loads(coco_path.read_text(encoding="utf-8"))
    names = {i["id"]: i["file_name"] for i in d["images"]}
    out = {}
    for a in d["annotations"]:
        k = np.array(a["keypoints"], dtype=np.float32).reshape(-1, 3)
        if k.shape[0] != 17:
            raise SystemExit(f"annotation {a['id']} has {k.shape[0]} keypoints, expected 17 (COCO order)")
        x, y, w, h = a["bbox"]
        out[Path(names[a["image_id"]]).name] = (k, (x, y, x + w, y + h))
    return out


def run_vitpose(model: OnnxVitPose, img: np.ndarray, box: tuple[float, ...]) -> np.ndarray:
    return model.predict(img, [box])[0]


def run_yolo(model, img: np.ndarray, box: tuple[float, ...]) -> np.ndarray | None:  # noqa: ANN001
    res = model.predict(img, verbose=False)[0]
    if res.keypoints is None or len(res.boxes) == 0:
        return None
    best, best_iou = None, 0.0
    for i, b in enumerate(res.boxes.xyxy.cpu().numpy()):
        v = iou(tuple(b), box)
        if v > best_iou:
            best, best_iou = i, v
    if best is None or best_iou < 0.3:
        return None
    xy = res.keypoints.xy[best].cpu().numpy()
    cf = res.keypoints.conf[best].cpu().numpy() if res.keypoints.conf is not None else np.ones(17)
    return np.concatenate([xy, cf[:, None]], axis=1)


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--gt", type=Path, required=True, help="CVAT export (COCO keypoints 1.0) json")
    ap.add_argument("--images", type=Path, required=True, help="folder with the images of the set")
    ap.add_argument("--manifest", type=Path, help="manifest.csv, to restrict to --split")
    ap.add_argument("--split", default="real_val")
    ap.add_argument("--group", default="", help="substring of image name defining a camera group, e.g. 'ddt'")
    ap.add_argument("--vitpose-onnx", type=str)
    ap.add_argument("--yolo-pose", type=str)
    ap.add_argument("--out", type=Path, default=Path("zero_shot_results.csv"))
    ap.add_argument("--boxes-from-manifest", action="store_true",
                    help="use the manifest person boxes (x1..y2) instead of the GT bbox. Use for CVAT exports, whose "
                         "bbox is fitted tightly around the annotated points")
    a = ap.parse_args()

    if not a.gt.is_file():
        raise SystemExit(f"GT file not found: {a.gt}. Export the corrected real_val frames from CVAT first "
                         "(see docs/POSE_ANNOTATION_GUIDE.md); this step cannot run before that.")
    gt = load_gt(a.gt)
    if a.boxes_from_manifest:
        if not a.manifest:
            raise SystemExit("--boxes-from-manifest needs --manifest")
        mboxes = {Path(r["image"]).name: tuple(float(r[k]) for k in ("x1", "y1", "x2", "y2"))
                  for r in csv.DictReader(open(a.manifest, encoding="utf-8"))}
        gt = {n: (k, mboxes[n]) for n, (k, _) in gt.items() if n in mboxes}
    names = set(gt)
    if a.manifest:
        names &= {Path(r["image"]).name for r in csv.DictReader(open(a.manifest, encoding="utf-8"))
                  if r["split"] == a.split}
    if a.group:
        names = {n for n in names if a.group in n}
    names = sorted(names)
    if not names:
        raise SystemExit("no frames selected: check --split / --group / file names")
    print(f"{len(names)} frames, split={a.split}, group={a.group or 'all'}")

    runners = {}
    if a.vitpose_onnx:
        m = OnnxVitPose(a.vitpose_onnx)
        runners["vitpose_swimxyz"] = lambda im, b: run_vitpose(m, im, b)
    if a.yolo_pose:
        from ultralytics import YOLO

        y = YOLO(a.yolo_pose)
        runners[Path(a.yolo_pose).stem] = lambda im, b: run_yolo(y, im, b)
    if not runners:
        raise SystemExit("give --vitpose-onnx and/or --yolo-pose")

    rows = []
    for mname, fn in runners.items():
        acc = {(s, t): [0, 0] for s in ("upper", "all") for t in THRS}
        n_conf3 = n_missing = 0
        times = []
        for n in names:
            img = cv2.imread(str(a.images / n))
            k, box = gt[n]
            t0 = time.perf_counter()
            pred = fn(img, box)
            times.append((time.perf_counter() - t0) * 1000)
            if pred is None:
                n_missing += 1
                pred = np.zeros((17, 3), dtype=np.float32)
                pred[:, :2] = -1e6
            n_conf3 += confident_upper(pred) >= 3
            for t in THRS:
                for s, idx in (("upper", UPPER_BODY), ("all", None)):
                    ok, tot = pck(pred, k, box, t, idx)
                    acc[(s, t)][0] += ok
                    acc[(s, t)][1] += tot
        row = {"model": mname, "frames": len(names), "no_prediction": n_missing,
               "frames_ge3_confident_upper": round(n_conf3 / len(names), 3),
               "latency_ms_median_per_person": round(float(np.median(times)), 1)}
        for (s, t), (ok, tot) in acc.items():
            row[f"PCK@{t}_{s}"] = round(ok / tot, 3) if tot else None
        rows.append(row)
        print(row)
    with open(a.out, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(rows[0]))
        wr.writeheader()
        wr.writerows(rows)
    print("wrote", a.out)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


Overwriting /content/pose_tools/zero_shot_eval.py


## 3. Data

In [6]:
# 3. Dataset (skipped if already unzipped in this session)
import shutil, zipfile
ROOT = Path('/content/pose_ds/datasets')
if not ROOT.is_dir():
    shutil.copyfile(DS_ZIP, '/content/pose_ds.zip')
    with zipfile.ZipFile('/content/pose_ds.zip') as z:
        z.extractall('/content/pose_ds')
print('frames:', sum(1 for _ in (ROOT / 'images').rglob('*.jpg')))


frames: 5613


---
# Part A: build the labelling package

In [107]:
# A1. Pick up to 25 of the 30 most disputed held-out frames, at least 8 frame numbers apart (no near-duplicates),
#     copy them with a manifest (person boxes from the dataset), zip to Drive for CVAT.
import csv, cv2
rows = list(csv.DictReader(open(DISPUTED)))
chosen = []
for r in rows:
    if all(abs(int(r['idx']) - int(c['idx'])) >= 8 for c in chosen):
        chosen.append(r)
    if len(chosen) == 25:
        break
SETD = Path('/content/e3_set')
shutil.rmtree(SETD, ignore_errors=True)
(SETD / 'images').mkdir(parents=True)
with open(SETD / 'manifest.csv', 'w', newline='') as f:
    wr = csv.writer(f)
    wr.writerow(['image', 'split', 'block', 'frame', 'x1', 'y1', 'x2', 'y2', 'disputed_keypoints'])
    for r in chosen:
        src = next(ROOT / 'images' / sp / r['file'] for sp in ('train', 'val') if (ROOT / 'images' / sp / r['file']).is_file())
        h, w = cv2.imread(str(src)).shape[:2]
        cx, cy, bw, bh = (float(v) for v in (ROOT / 'labels' / src.parent.name / (src.stem + '.txt')).read_text().split()[1:5])
        box = [(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h]
        shutil.copyfile(src, SETD / 'images' / r['file'])
        wr.writerow([f"images/{r['file']}", 'e3', r['block'], r['idx'], *[round(v, 1) for v in box], r['disputed_keypoints']])
ZIP = E3 / 'e3_set.zip'
!cd /content && rm -f "{ZIP}" && zip -qr "{ZIP}" e3_set
print(len(chosen), 'frames ->', ZIP)
for r in chosen:
    print(f"  {r['file']}  block {r['block']}  disputed keypoints {r['disputed_keypoints']}")


14 frames -> /content/drive/MyDrive/pose_baseline/e3/e3_set.zip
  frame_50232.jpg  block 16  disputed keypoints 11
  frame_07996.jpg  block 2  disputed keypoints 10
  frame_08024.jpg  block 2  disputed keypoints 10
  frame_08064.jpg  block 2  disputed keypoints 10
  frame_08128.jpg  block 2  disputed keypoints 10
  frame_08150.jpg  block 2  disputed keypoints 10
  frame_47155.jpg  block 15  disputed keypoints 10
  frame_50192.jpg  block 16  disputed keypoints 10
  frame_08054.jpg  block 2  disputed keypoints 9
  frame_08141.jpg  block 2  disputed keypoints 9
  frame_47218.jpg  block 15  disputed keypoints 9
  frame_50264.jpg  block 16  disputed keypoints 9
  frame_50299.jpg  block 16  disputed keypoints 9
  frame_08192.jpg  block 2  disputed keypoints 8


**Manual step (a person, about 1 hour):** in CVAT, create a new task in project `pose_swimmer_coco17` with the images from `e3_set.zip` (no annotation import). Draw one `person` skeleton per frame from scratch. Mark points you cannot see as not visible; do not guess. Export COCO Keypoints 1.0 and upload it as `MyDrive/pose_baseline/e3/pose_gt_e3.json`.

---
# Part B: evaluation against human labels

In [7]:
# B0. After labelling: restore the set if needed and check the ground-truth / reference file
import json, glob, zipfile
SETD = Path('/content/e3_set')
if not SETD.is_dir():
    with zipfile.ZipFile(E3 / 'e3_set.zip') as z:
        z.extractall('/content')
GT = E3 / 'pose_gt_e3.json'
if not GT.is_file():
    # files uploaded in the Drive web page can take a while to appear in Colab: refresh the mount once
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
if not GT.is_file():
    found = sorted(set(glob.glob('/content/drive/MyDrive/**/pose_gt_e3*.json', recursive=True)))
    print('Not at', GT)
    print('Files named pose_gt_e3*.json found in MyDrive:', *found or ['none'], sep=chr(10) + '  ')
    print('Contents of', E3, ':', sorted(p.name for p in E3.iterdir()))
    raise SystemExit('Move or rename the uploaded file to ' + str(GT) + ' and run this cell again')
g = json.loads(GT.read_text())
names = {i['id']: i['file_name'] for i in g['images']}
labelled = [a for a in g['annotations'] if any(v > 0 for v in a['keypoints'][2::3])]
print('provenance:', g.get('info', {}).get('provenance', 'not stated (treated as human)'))
print('GT images:', len(g['images']), '| skeletons with >= 1 labelled point:', len(labelled))
print('keypoints per skeleton:', {len(a['keypoints']) // 3 for a in g['annotations']})


provenance: ai_visual_review_unverified
GT images: 14 | skeletons with >= 1 labelled point: 14
keypoints per skeleton: {17}


In [12]:
# B1. HUMAN ground truth: ViTPose-L SwimXYZ vs pretrained YOLO11s-pose vs distilled YOLO11s-pose
#     (same frames, same person boxes from the manifest, same metric)
!cd /content/pose_tools && python zero_shot_eval.py --gt "{GT}" --images "{SETD}/images" --manifest "{SETD}/manifest.csv" --split e3 --boxes-from-manifest --vitpose-onnx "{ONNX}" --yolo-pose "{YOLO_PT}" --out "{E3}/e3_vitpose_and_pretrained.csv"
!cd /content/pose_tools && python zero_shot_eval.py --gt "{GT}" --images "{SETD}/images" --manifest "{SETD}/manifest.csv" --split e3 --boxes-from-manifest --yolo-pose "{DIST_PT}" --out "{E3}/e3_distilled.csv"
import pandas as pd
res = pd.concat([pd.read_csv(E3 / 'e3_vitpose_and_pretrained.csv'), pd.read_csv(E3 / 'e3_distilled.csv').assign(model='distilled')])
res.to_csv(E3 / 'e3_comparison.csv', index=False)
prov = g.get('info', {}).get('provenance', 'human')
if prov == 'human':
    print('ACCURACY AGAINST HUMAN LABELS (hardest, most-disputed held-out frames)')
else:
    print(f'AGREEMENT WITH AI REFERENCE LABELS ({prov}) - NOT HUMAN GROUND TRUTH, NOT ACCURACY')
print(res.set_index('model').T.to_string())


14 frames, split=e3, group=all
{'model': 'vitpose_swimxyz', 'frames': 14, 'no_prediction': 0, 'frames_ge3_confident_upper': 1.0, 'latency_ms_median_per_person': 36.3, 'frames_better_with_lr_swap': 4, 'PCK@0.05_upper': 0.514, 'PCK@0.1_upper': 0.703, 'PCK@0.2_upper': 0.73, 'PCK@0.05_all': 0.369, 'PCK@0.1_all': 0.615, 'PCK@0.2_all': 0.692, 'PCK@0.05_upper_swaptol': 0.554, 'PCK@0.1_upper_swaptol': 0.77, 'PCK@0.2_upper_swaptol': 0.851, 'PCK@0.05_all_swaptol': 0.392, 'PCK@0.1_all_swaptol': 0.654, 'PCK@0.2_all_swaptol': 0.762}
{'model': 'yolo11s-pose', 'frames': 14, 'no_prediction': 0, 'frames_ge3_confident_upper': 1.0, 'latency_ms_median_per_person': 12.2, 'frames_better_with_lr_swap': 10, 'PCK@0.05_upper': 0.068, 'PCK@0.1_upper': 0.176, 'PCK@0.2_upper': 0.243, 'PCK@0.05_all': 0.054, 'PCK@0.1_all': 0.146, 'PCK@0.2_all': 0.315, 'PCK@0.05_upper_swaptol': 0.108, 'PCK@0.1_upper_swaptol': 0.311, 'PCK@0.2_upper_swaptol': 0.541, 'PCK@0.05_all_swaptol': 0.185, 'PCK@0.1_all_swaptol': 0.4, 'PCK@0.2_al

In [10]:
!grep -c swaptol /content/pose_tools/zero_shot_eval.py

0


In [11]:
from pathlib import Path
p = Path('/content/pose_tools/zero_shot_eval.py')
s = p.read_text()
if 'swaptol' not in s:
    s = s.replace('THRS = (0.05, 0.1, 0.2)',
        'THRS = (0.05, 0.1, 0.2)\nFLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]')
    s = s.replace('''        acc = {(s, t): [0, 0] for s in ("upper", "all") for t in THRS}
        n_conf3 = n_missing = 0''', '''        acc = {(s, t): [0, 0] for s in ("upper", "all") for t in THRS}
        acc_sw = {(s, t): [0, 0] for s in ("upper", "all") for t in THRS}
        n_conf3 = n_missing = n_swapped = 0''')
    s = s.replace('''            for t in THRS:
                for s, idx in (("upper", UPPER_BODY), ("all", None)):
                    ok, tot = pck(pred, k, box, t, idx)
                    acc[(s, t)][0] += ok
                    acc[(s, t)][1] += tot''', '''            swapped = pred[FLIP_IDX]
            use_swap = pck(swapped, k, box, 0.2)[0] > pck(pred, k, box, 0.2)[0]
            n_swapped += use_swap
            best = swapped if use_swap else pred
            for t in THRS:
                for s, idx in (("upper", UPPER_BODY), ("all", None)):
                    ok, tot = pck(pred, k, box, t, idx)
                    acc[(s, t)][0] += ok
                    acc[(s, t)][1] += tot
                    ok2, _ = pck(best, k, box, t, idx)
                    acc_sw[(s, t)][0] += ok2
                    acc_sw[(s, t)][1] += tot''')
    s = s.replace('''               "latency_ms_median_per_person": round(float(np.median(times)), 1)}''',
        '''               "latency_ms_median_per_person": round(float(np.median(times)), 1),
               "frames_better_with_lr_swap": n_swapped}''')
    s = s.replace('''            row[f"PCK@{t}_{s}"] = round(ok / tot, 3) if tot else None''',
        '''            row[f"PCK@{t}_{s}"] = round(ok / tot, 3) if tot else None
        for (s, t), (ok, tot) in acc_sw.items():
            row[f"PCK@{t}_{s}_swaptol"] = round(ok / tot, 3) if tot else None''')
    p.write_text(s)
compile(p.read_text(), str(p), 'exec')
print('swaptol occurrences:', p.read_text().count('swaptol'), '| FLIP_IDX present:', 'FLIP_IDX =' in p.read_text())

swaptol occurrences: 1 | FLIP_IDX present: True
